**Вариант 10**

**Практическая работа 3. Работа с реальными данными**

Цель работы: научиться читать реальные файлы разных форматов (CSV, Excel, JSON), объединять несколько источников в один `DataFrame`, заполнять пропуски и сохранять результат в формате parquet.

Задание варианта 10:

1. считать все доступные датасеты: CSV — заказы, Excel — товары, JSON — события (оба формата);
2. построить единый `DataFrame`, в котором для каждого пользователя есть количество заказов, суммарная выручка, количество событий, первый и последний момент активности по событиям и по заказам;
3. заполнить отсутствующие значения нулями или `NaN` по необходимости;
4. сохранить итоговый `DataFrame` в `user_profile.parquet`;
5. оценить, сколько памяти занимает итоговая таблица.

Исходные данные

Файлы из архива практической работы лежат в папке `for_lab_3`:

| Файл | Формат | Содержимое |
|---|---|---|
| `orders_2024_01.csv`, `orders_2024_02.csv`, `orders_2024_03.csv` | CSV | заказы интернет-магазина по месяцам |
| `products.xlsx` | Excel, листы `products` и `categories` | справочник товаров и категорий |
| `events_records.json` | JSON (records) | события/клики пользователей |
| `events_lines.json` | JSON Lines | события/клики пользователей |

Структура заказов: `order_id`, `user_id`, `order_datetime`, `product_id`, `price`, `quantity`, `status`.

Структура событий: `event_id`, `user_id`, `event_time`, `event_type`, `page`.

In [1]:
import glob
import json
import os

import numpy as np
import pandas as pd

# папка с исходными файлами практической работы
DATA_DIR = 'for_lab_3'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = 'data'

print('Рабочая папка с данными:', os.path.abspath(DATA_DIR))
print()
for path in sorted(glob.glob(os.path.join(DATA_DIR, '*'))):
    print(f'{os.path.basename(path):<25}{os.path.getsize(path):>10} байт')

Рабочая папка с данными: D:\учёба\vsuet\ims\for_lab_3

events_lines.json            239555 байт
events_records.json          224211 байт
orders_2024_01.csv           244027 байт
orders_2024_02.csv           245248 байт
orders_2024_03.csv           250278 байт
products.xlsx                  6526 байт
user_profile.parquet          11919 байт


**1. Считать все доступные датасеты: CSV — заказы, Excel — товары, JSON — события (оба формата)**

In [2]:
# типы данных для оптимизации памяти
orders_dtypes = {
    'order_id': 'int64',
    'user_id': 'int32',
    'product_id': 'int32',
    'price': 'float64',
    'quantity': 'int16',
    'status': 'category',
}

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, 'orders_2024_*.csv')))
orders_parts = []

for path in csv_files:
    part = pd.read_csv(
        path,
        dtype=orders_dtypes,
        parse_dates=['order_datetime'],
    )
    orders_parts.append(part)
    print(f'[+] {os.path.basename(path)}: {part.shape[0]} строк, {part.shape[1]} столбцов')

# объединяем все месяцы в одну таблицу
orders = pd.concat(orders_parts, ignore_index=True)

# выручка по каждой позиции заказа
orders['revenue'] = orders['price'] * orders['quantity']

print()
print('Всего строк с заказами:', len(orders))
print('Уникальных заказов:    ', orders['order_id'].nunique())
print('Период заказов:        ', orders['order_datetime'].min(), '—', orders['order_datetime'].max())
print()
print(orders.head().to_string())
print()
print(orders.dtypes.to_string())

[+] orders_2024_01.csv: 5000 строк, 7 столбцов
[+] orders_2024_02.csv: 5000 строк, 7 столбцов
[+] orders_2024_03.csv: 5000 строк, 7 столбцов

Всего строк с заказами: 15000
Уникальных заказов:     15000
Период заказов:         2024-01-01 00:10:25 — 2024-03-31 23:34:00

   order_id  user_id      order_datetime  product_id     price  quantity    status   revenue
0         1     1044 2024-01-03 18:39:26          26  14038.15         1  refunded  14038.15
1         2     1079 2024-01-03 17:45:43          16   9172.60         1      paid   9172.60
2         3     1041 2024-01-06 22:46:38          25   9035.26         2      paid  18070.52
3         4     1083 2024-01-08 21:03:51           9  15234.33         3      paid  45702.99
4         5     1068 2024-01-03 23:26:53          24   4134.38         2      paid   8268.76

order_id                   int64
user_id                    int32
order_datetime    datetime64[us]
product_id                 int32
price                    float64
quantit

In [3]:
excel_path = os.path.join(DATA_DIR, 'products.xlsx')

# читаем сразу два листа книги
sheets = pd.read_excel(excel_path, sheet_name=['products', 'categories'])
products = sheets['products']
categories = sheets['categories']

print('Лист products:  ', products.shape)
print('Лист categories:', categories.shape)
print()
print(products.head().to_string())
print()
print(categories.to_string())

# расширяем справочник товаров описанием категории
products_full = products.merge(
    categories[['category_id', 'description']],
    on='category_id',
    how='left',
)

print()
print('Расширенный справочник товаров:')
print(products_full.head().to_string())

Лист products:   (30, 5)
Лист categories: (5, 3)

   product_id product_name  category_id category_name  weight_kg
0           1    Product 1            1   Electronics      11.55
1           2    Product 2            3         Sport       2.22
2           3    Product 3            5          Toys      11.43
3           4    Product 4            1   Electronics       4.17
4           5    Product 5            5          Toys       1.81

   category_id category_name                  description
0            1   Electronics  Description for Electronics
1            2          Home         Description for Home
2            3         Sport        Description for Sport
3            4       Fashion      Description for Fashion
4            5          Toys         Description for Toys

Расширенный справочник товаров:
   product_id product_name  category_id category_name  weight_kg                  description
0           1    Product 1            1   Electronics      11.55  Description for El

In [4]:
def read_events_records(path):
    '''Читает JSON в формате records (список объектов).'''
    with open(path, encoding='utf-8') as f:
        raw = json.load(f)

    df = pd.DataFrame(raw)

    # файл может быть сохранён транспонированным: 5 строк-признаков и 2000 столбцов
    if 'event_id' not in df.columns:
        df = df.T.reset_index(drop=True)
        df.columns = ['event_id', 'user_id', 'event_time', 'event_type', 'page']

    return df


# --- формат records ---
events_records = read_events_records(os.path.join(DATA_DIR, 'events_records.json'))
events_records['event_time'] = pd.to_datetime(events_records['event_time'])
events_records['event_id'] = events_records['event_id'].astype('int64')
events_records['user_id'] = events_records['user_id'].astype('int32')

# --- формат JSON Lines ---
events_lines = pd.read_json(os.path.join(DATA_DIR, 'events_lines.json'), lines=True)
events_lines['event_time'] = pd.to_datetime(events_lines['event_time'])
events_lines['event_id'] = events_lines['event_id'].astype('int64')
events_lines['user_id'] = events_lines['user_id'].astype('int32')

print('events_records.json:', events_records.shape)
print(events_records.head(3).to_string())
print()
print('events_lines.json:  ', events_lines.shape)
print(events_lines.head(3).to_string())

# объединяем оба источника событий
events_all = pd.concat([events_records, events_lines], ignore_index=True)

print()
print('Всего событий:      ', len(events_all))
print('Период событий:     ', events_all['event_time'].min(), '—', events_all['event_time'].max())
print('Типы событий:')
print(events_all['event_type'].value_counts().to_string())

events_records.json: (2000, 5)
   event_id  user_id          event_time   event_type         page
0         1     1085 2024-01-30 20:37:32    page_view  /product/26
1         2     1006 2024-02-24 22:55:13  add_to_cart  /product/28
2         3     1024 2024-02-14 17:49:43        click  /product/27

events_lines.json:   (2000, 5)
   event_id  user_id          event_time   event_type         page
0      2001     1029 2024-03-19 00:34:38     purchase  /product/20
1      2002     1056 2024-03-25 17:16:08  add_to_cart        /cart
2      2003     1081 2024-02-06 06:09:49  add_to_cart  /product/30

Всего событий:       4000
Период событий:      2024-01-01 01:06:07 — 2024-03-31 23:39:39
Типы событий:
event_type
click          819
purchase       807
page_view      804
scroll         788
add_to_cart    782


**2. Построить единый DataFrame: количество заказов, суммарная выручка, количество событий, первый и последний момент активности**

In [5]:
# количество заказов, выручка и границы активности по заказам
orders_by_user = orders.groupby('user_id').agg(
    orders_count=('order_id', 'nunique'),
    total_revenue=('revenue', 'sum'),
    first_order_datetime=('order_datetime', 'min'),
    last_order_datetime=('order_datetime', 'max'),
)

# число уникальных товаров в заказах пользователя
unique_products = orders.groupby('user_id')['product_id'].nunique().rename('unique_products')

# подключаем справочник товаров из Excel и считаем выручку по категориям
orders_enriched = orders.merge(
    products[['product_id', 'category_name']],
    on='product_id',
    how='left',
)

category_revenue = (
    orders_enriched
    .groupby(['user_id', 'category_name'], observed=True)['revenue']
    .sum()
    .reset_index()
)

# любимая категория — та, что принесла пользователю наибольшую выручку
top_category = (
    category_revenue
    .sort_values('revenue', ascending=False)
    .drop_duplicates('user_id')[['user_id', 'category_name']]
    .rename(columns={'category_name': 'top_category'})
)

print('Пользователей с заказами:', orders_by_user.shape[0])
print()
print(orders_by_user.head().to_string())
print()
print('Выручка по категориям (первые 5 строк):')
print(category_revenue.head().to_string())

Пользователей с заказами: 100

         orders_count  total_revenue first_order_datetime last_order_datetime
user_id                                                                      
1001              177     4435767.09  2024-01-02 02:51:37 2024-03-31 21:32:19
1002              123     2830004.52  2024-01-02 12:57:34 2024-03-31 19:45:16
1003              156     4025266.18  2024-01-02 02:02:44 2024-03-31 19:36:26
1004              147     3788030.56  2024-01-01 00:19:34 2024-03-31 20:01:40
1005              141     3185653.37  2024-01-01 01:11:51 2024-03-31 18:16:10

Выручка по категориям (первые 5 строк):
   user_id category_name     revenue
0     1001   Electronics  1079366.81
1     1001       Fashion   394191.22
2     1001          Home   510953.03
3     1001         Sport  1212972.66
4     1001          Toys  1238283.37


In [6]:
# количество событий и границы активности по событиям
events_by_user = events_all.groupby('user_id').agg(
    events_count=('event_id', 'count'),
    first_event_time=('event_time', 'min'),
    last_event_time=('event_time', 'max'),
)

print('Пользователей с событиями:', events_by_user.shape[0])
print()
print(events_by_user.head().to_string())

Пользователей с событиями: 100

         events_count    first_event_time     last_event_time
user_id                                                      
1001               54 2024-01-01 06:25:31 2024-03-31 22:29:02
1002               41 2024-01-04 12:58:00 2024-03-31 21:24:08
1003               39 2024-01-09 20:43:11 2024-03-29 20:53:24
1004               37 2024-01-04 14:36:56 2024-03-27 04:58:11
1005               44 2024-01-01 16:44:30 2024-03-30 04:58:21


In [7]:
# все пользователи из заказов и событий
all_users = np.union1d(orders['user_id'].unique(), events_all['user_id'].unique())
user_profile = pd.DataFrame({'user_id': pd.Series(all_users, dtype='int32')})

# последовательно добавляем агрегаты из разных источников
user_profile = user_profile.merge(orders_by_user, on='user_id', how='left')
user_profile = user_profile.merge(unique_products, on='user_id', how='left')
user_profile = user_profile.merge(top_category, on='user_id', how='left')
user_profile = user_profile.merge(events_by_user, on='user_id', how='left')

# сортируем по идентификатору пользователя
user_profile = user_profile.sort_values('user_id').reset_index(drop=True)

print('Размер итоговой таблицы:', user_profile.shape)
print()
print(user_profile.head(10).to_string())
print()
print('Пропуски до заполнения:')
print(user_profile.isna().sum().to_string())

Размер итоговой таблицы: (100, 10)

   user_id  orders_count  total_revenue first_order_datetime last_order_datetime  unique_products top_category  events_count    first_event_time     last_event_time
0     1001           177     4435767.09  2024-01-02 02:51:37 2024-03-31 21:32:19               30         Toys            54 2024-01-01 06:25:31 2024-03-31 22:29:02
1     1002           123     2830004.52  2024-01-02 12:57:34 2024-03-31 19:45:16               29         Toys            41 2024-01-04 12:58:00 2024-03-31 21:24:08
2     1003           156     4025266.18  2024-01-02 02:02:44 2024-03-31 19:36:26               30         Toys            39 2024-01-09 20:43:11 2024-03-29 20:53:24
3     1004           147     3788030.56  2024-01-01 00:19:34 2024-03-31 20:01:40               30        Sport            37 2024-01-04 14:36:56 2024-03-27 04:58:11
4     1005           141     3185653.37  2024-01-01 01:11:51 2024-03-31 18:16:10               30         Toys            44 2024-01-01 16:

**3. Заполнить отсутствующие значения нулями или NaN по необходимости**

In [8]:
# числовые показатели: нет заказов/событий — значит 0
zero_columns = ['orders_count', 'total_revenue', 'unique_products', 'events_count']
user_profile[zero_columns] = user_profile[zero_columns].fillna(0)

# текстовый признак
user_profile['top_category'] = user_profile['top_category'].fillna('нет заказов')

# аккуратные типы после заполнения
user_profile['orders_count'] = user_profile['orders_count'].astype('int32')
user_profile['events_count'] = user_profile['events_count'].astype('int32')
user_profile['unique_products'] = user_profile['unique_products'].astype('int32')

# даты остаются NaT — это корректное обозначение отсутствия активности
print(user_profile.dtypes.to_string())
print()
print('Пропуски после заполнения:')
print(user_profile.isna().sum().to_string())
print()
print('Пользователей всего:  ', len(user_profile))
print('С заказами:           ', int((user_profile['orders_count'] > 0).sum()))
print('С событиями:          ', int((user_profile['events_count'] > 0).sum()))
print('Без заказов (только события):', int((user_profile['orders_count'] == 0).sum()))
print('Без событий (только заказы): ', int((user_profile['events_count'] == 0).sum()))

user_id                          int32
orders_count                     int32
total_revenue                  float64
first_order_datetime    datetime64[us]
last_order_datetime     datetime64[us]
unique_products                  int32
top_category                       str
events_count                     int32
first_event_time        datetime64[us]
last_event_time         datetime64[us]

Пропуски после заполнения:
user_id                 0
orders_count            0
total_revenue           0
first_order_datetime    0
last_order_datetime     0
unique_products         0
top_category            0
events_count            0
first_event_time        0
last_event_time         0

Пользователей всего:   100
С заказами:            100
С событиями:           100
Без заказов (только события): 0
Без событий (только заказы):  0


**4. Сохранить итоговый DataFrame в user_profile.parquet**

In [9]:
parquet_path = os.path.join(DATA_DIR, 'user_profile.parquet')

user_profile.to_parquet(parquet_path, index=False)

print('Файл сохранён:', os.path.abspath(parquet_path))
print(f'Размер файла:  {os.path.getsize(parquet_path)} байт '
      f'({os.path.getsize(parquet_path) / 1024:.1f} КБ)')

# проверка обратного чтения
check = pd.read_parquet(parquet_path)
print()
print('Прочитано обратно:', check.shape)
print(check.head(5).to_string())

Файл сохранён: D:\учёба\vsuet\ims\for_lab_3\user_profile.parquet
Размер файла:  11919 байт (11.6 КБ)

Прочитано обратно: (100, 10)
   user_id  orders_count  total_revenue first_order_datetime last_order_datetime  unique_products top_category  events_count    first_event_time     last_event_time
0     1001           177     4435767.09  2024-01-02 02:51:37 2024-03-31 21:32:19               30         Toys            54 2024-01-01 06:25:31 2024-03-31 22:29:02
1     1002           123     2830004.52  2024-01-02 12:57:34 2024-03-31 19:45:16               29         Toys            41 2024-01-04 12:58:00 2024-03-31 21:24:08
2     1003           156     4025266.18  2024-01-02 02:02:44 2024-03-31 19:36:26               30         Toys            39 2024-01-09 20:43:11 2024-03-29 20:53:24
3     1004           147     3788030.56  2024-01-01 00:19:34 2024-03-31 20:01:40               30        Sport            37 2024-01-04 14:36:56 2024-03-27 04:58:11
4     1005           141     3185653.37  202

**5. Оценить, сколько памяти занимает итоговая таблица**

In [10]:
memory = user_profile.memory_usage(deep=True)

print('Память по столбцам (байт):')
print(memory.to_string())
print()
print(f'Итого в памяти:  {memory.sum()} байт = {memory.sum() / 1024:.1f} КБ')
print(f'На пользователя: {memory.sum() / len(user_profile):.1f} байт')

# для сравнения сохраняем те же данные в CSV и сравниваем размеры файлов
csv_path = os.path.join(DATA_DIR, 'user_profile.csv')
user_profile.to_csv(csv_path, index=False)
csv_size = os.path.getsize(csv_path)
parquet_size = os.path.getsize(parquet_path)
os.remove(csv_path)  # временный файл, нужен только для сравнения

print(f'Файл parquet:    {parquet_size} байт = {parquet_size / 1024:.1f} КБ')
print(f'Файл CSV:        {csv_size} байт = {csv_size / 1024:.1f} КБ')
print(f'Отношение parquet/CSV: {parquet_size / csv_size:.2f}')
print()
print('Информация о таблице:')
user_profile.info(memory_usage='deep')

Память по столбцам (байт):
Index                    132
user_id                  400
orders_count             400
total_revenue            800
first_order_datetime     800
last_order_datetime      800
unique_products          400
top_category            1264
events_count             400
first_event_time         800
last_event_time          800

Итого в памяти:  6996 байт = 6.8 КБ
На пользователя: 70.0 байт
Файл parquet:    11919 байт = 11.6 КБ
Файл CSV:        11424 байт = 11.2 КБ
Отношение parquet/CSV: 1.04

Информация о таблице:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   user_id               100 non-null    int32         
 1   orders_count          100 non-null    int32         
 2   total_revenue         100 non-null    float64       
 3   first_order_datetime  100 non-null    datetime64[us]
 4   last_order_dateti

**Проверка результата**

In [11]:
print('Топ-5 пользователей по выручке:')
print(
    user_profile
    .nlargest(5, 'total_revenue')
    [['user_id', 'orders_count', 'total_revenue', 'events_count', 'top_category']]
    .to_string(index=False)
)

print()
print('Топ-5 пользователей по количеству событий:')
print(
    user_profile
    .nlargest(5, 'events_count')
    [['user_id', 'events_count', 'orders_count', 'total_revenue']]
    .to_string(index=False)
)

Топ-5 пользователей по выручке:
 user_id  orders_count  total_revenue  events_count top_category
    1066           164     4769058.75            43        Sport
    1081           169     4715929.48            34         Toys
    1012           171     4613593.65            39         Toys
    1017           163     4558872.88            34        Sport
    1027           171     4557347.85            36         Toys

Топ-5 пользователей по количеству событий:
 user_id  events_count  orders_count  total_revenue
    1091            64           147     4204414.64
    1051            59           153     4117942.24
    1072            57           150     3958219.27
    1093            55           133     3260214.56
    1001            54           177     4435767.09
